# Lesson 6 — Policy and guardrails

**You will:** classify tools by risk and side effects, block dangerous ones, and require approval for risky ones.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/06-policy-and-guardrails/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

An LLM will happily call any tool you give it. If you give an agent a `delete_user_account` tool, you cannot rely on the model to decide *not* to call it. Safety has to live somewhere the model cannot override.

In BareBear, that's **`Policy`**. Every tool call passes through policy *before* it runs. The mental model: a tool is a *capability*, policy is the *permission system*.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")

# Optional: pin a specific free-tier model. If unset, barebear uses its default.
# Free-tier model availability rotates — swap here if a lesson cell errors with
# "model not available". Any *:free model on https://openrouter.ai/models works.
# os.environ["BAREBEAR_MODEL"] = "meta-llama/llama-3.2-3b-instruct:free"

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")
print("Model:", os.environ.get("BAREBEAR_MODEL", "(framework default)"))

## A blocked tool the model wants to call

In [ ]:
from barebear import Bear, Task, Tool, Policy, OpenRouterModel

def delete_everything() -> str:
    # This function will NOT run — policy blocks the call before execution.
    return "Deleted."

policy = Policy(blocked_tools=["delete_everything"])

bear = Bear(
    model=OpenRouterModel(),
    tools=[Tool(name="delete_everything", fn=delete_everything,
                description="Delete all user data. High risk.", risk="high")],
    policy=policy,
)

report = bear.run(Task(goal="Reset the system to factory state."), trace=True)
print()
print(report.summary())

## What just happened

Look at the trace: you'll see the model *requested* the dangerous tool, but the framework recorded a `policy_block` step instead of executing it. The function `delete_everything` was never called. The agent then either continued without that capability or returned a final answer.

This is the safety pattern: **the model can want anything; policy decides what it can do.**

## Side effects

Tools declare their `side_effects` as `none`, `internal`, or `external`. Policy can reject any tool with external side effects:

In [ ]:
def hit_external_api(endpoint: str) -> str:
    return f"(would call {endpoint})"

strict = Policy(allow_external_side_effects=False)

bear = Bear(
    model=OpenRouterModel(),
    tools=[Tool(name="hit_external_api", fn=hit_external_api,
                description="Call an external service",
                side_effects="external")],
    policy=strict,
)

report = bear.run(Task(goal="Trigger the deploy webhook."))
print(report.summary())

## Exercise

1. Switch `blocked_tools=["delete_everything"]` for `require_approval_for=["delete_everything"]`. Run again. The agent should pause with `status='paused'`. (Lesson 7 covers what to do then.)
2. Add a tool with `risk='high'` and `side_effects='external'`. Configure a policy that allows it only with approval.

## What's next

[Lesson 7 →](../07-checkpoints/lesson.ipynb)